In [14]:
import time
import pandas as pd
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim


In [11]:
! pip install fvcore

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=3a8eb84f81d64edc9b9744704284c3b1a3a5aeb8461a8e647392a69893929a21
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for iopath: filename=iopath-0.1.10-py3-none-any.whl size=31527 sha256=9b82afc453fcaed9d1860b07b25bd665969c97145b6271d57d7afbd9e88f7f6c
  Stored in directory: /root/.cache/pip/wheels/7c/96/04/4f5f31ff812f684f69f40cb1634357812220aac58d4698048c
Successfully built fvcore iopath


In [4]:
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader

def get_dataloaders(dataset_name, batch_size=16):
    if dataset_name == "FashionMNIST":
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

        train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

        # Split train_dataset into training and validation sets
        train_size = int(0.8 * len(train_dataset))
        val_size = len(train_dataset) - train_size
        train_dataset, val_dataset = torch.utils.data.random_split(train_dataset, [train_size, val_size])

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    else:
        raise ValueError(f"Unsupported dataset: {dataset_name}")

    return train_loader, val_loader, test_loader

In [5]:
def train_model(model, train_loader, val_loader, optimizer, epochs, device):
    criterion = nn.CrossEntropyLoss()
    model.to(device)

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Optional: Validation step
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {running_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}, Val Accuracy: {100 * correct / total:.2f}%")

In [7]:
def evaluate_model(model, test_loader, device):
    model.to(device)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    return accuracy

In [12]:

# for batch size 16

def estimate_flops(model, input_size=(1, 3, 64, 64)):
    try:
        from fvcore.nn import FlopCountAnalysis
        dummy = torch.randn(input_size).to(next(model.parameters()).device)
        flops = FlopCountAnalysis(model, dummy).total()
        return round(flops / 1e6, 2)  # MFLOPs
    except:
        return None


def train_with_timing(model, train_loader, val_loader, optimizer, device, epochs=2):
    start = time.time()

    train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        epochs=epochs,
        device=device
    )

    end = time.time()
    return round((end - start) * 1000, 2)

def run_q2_experiments(epochs=2):
    configs = [
        ("CPU",  "SGD",  0.001),
        ("CPU",  "Adam", 0.001),
        ("GPU",  "SGD",  0.001),
        ("GPU",  "Adam", 0.001),
    ]

    results = []

    for compute, opt_name, lr in configs:
        device_used = torch.device(
            "cuda" if compute == "GPU" and torch.cuda.is_available() else "cpu"
        )

        print(f"\nRunning on {compute} with {opt_name}")

        train_loader, val_loader, test_loader = get_dataloaders(
            "FashionMNIST", batch_size=16
        )

        for model_name in ["resnet18", "resnet50"]:

            # No pretrained weights
            model = models.resnet18(weights=None) if model_name == "resnet18" \
                    else models.resnet50(weights=None)

            # Modify the first convolutional layer to accept 1 input channel for grayscale images
            model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            model.fc = nn.Linear(model.fc.in_features, 10)

            optimizer = (
                optim.SGD(model.parameters(), lr=lr)
                if opt_name == "SGD"
                else optim.Adam(model.parameters(), lr=lr)
            )

            # Measure training time
            train_time = train_with_timing(
                model,
                train_loader,
                val_loader,
                optimizer,
                device=device_used,
                epochs=epochs
            )

            # Accuracy
            test_acc = evaluate_model(model, test_loader, device=device_used)

            # FLOPs
            flops = estimate_flops(model, input_size=(1, 1, 64, 64)) # Adjusted input size for FLOPs

            results.append({
                "Compute": compute,
                "Batch Size": 16,
                "Optimizer": opt_name,
                "Learning Rate": lr,
                "Model": model_name,
                "Test Accuracy (%)": round(test_acc, 2),
                "Train Time (ms)": train_time,
                "FLOPs (MFLOPs)": flops
            })

            print(f"{model_name} | Acc={test_acc:.2f}% | Time={train_time} ms | FLOPs={flops}")

    return pd.DataFrame(results)

In [13]:
df_q2 = run_q2_experiments(epochs=2)
print("\nFinal Q2 Results:")
print(df_q2)




Running on CPU with SGD
Epoch 1/2, Train Loss: 0.6316, Val Loss: 0.4112, Val Accuracy: 85.03%
Epoch 2/2, Train Loss: 0.4099, Val Loss: 0.3538, Val Accuracy: 86.94%


resnet18 | Acc=86.18% | Time=1417608.87 ms | FLOPs=142.04
Epoch 1/2, Train Loss: 1.0038, Val Loss: 0.5905, Val Accuracy: 77.53%
Epoch 2/2, Train Loss: 0.6451, Val Loss: 0.4876, Val Accuracy: 81.91%


resnet50 | Acc=81.41% | Time=3153081.4 ms | FLOPs=329.06

Running on CPU with Adam
Epoch 1/2, Train Loss: 0.5196, Val Loss: 0.4852, Val Accuracy: 82.49%
Epoch 2/2, Train Loss: 0.3695, Val Loss: 0.3204, Val Accuracy: 88.56%


resnet18 | Acc=87.54% | Time=2023533.34 ms | FLOPs=142.04
Epoch 1/2, Train Loss: 0.8369, Val Loss: 0.5461, Val Accuracy: 81.10%
Epoch 2/2, Train Loss: 0.6187, Val Loss: 0.4009, Val Accuracy: 85.03%


resnet50 | Acc=83.62% | Time=4042739.54 ms | FLOPs=329.06

Running on GPU with SGD
Epoch 1/2, Train Loss: 0.6343, Val Loss: 0.4041, Val Accuracy: 84.96%
Epoch 2/2, Train Loss: 0.4139, Val Loss: 0.3458, Val Accuracy: 87.10%


resnet18 | Acc=85.81% | Time=1317546.8 ms | FLOPs=142.04
Epoch 1/2, Train Loss: 1.0670, Val Loss: 0.6312, Val Accuracy: 77.40%
Epoch 2/2, Train Loss: 0.6737, Val Loss: 0.5367, Val Accuracy: 80.17%


resnet50 | Acc=78.65% | Time=3004023.19 ms | FLOPs=329.06

Running on GPU with Adam
Epoch 1/2, Train Loss: 0.5165, Val Loss: 0.3390, Val Accuracy: 87.27%
Epoch 2/2, Train Loss: 0.3638, Val Loss: 0.3401, Val Accuracy: 87.81%


resnet18 | Acc=87.00% | Time=2017233.77 ms | FLOPs=142.04
Epoch 1/2, Train Loss: 0.8883, Val Loss: 0.5640, Val Accuracy: 79.69%
Epoch 2/2, Train Loss: 0.5986, Val Loss: 0.4458, Val Accuracy: 83.74%


resnet50 | Acc=82.55% | Time=4216234.48 ms | FLOPs=329.06

Final Q2 Results:
  Compute  Batch Size Optimizer  Learning Rate     Model  Test Accuracy (%)  \
0     CPU          16       SGD          0.001  resnet18              86.18   
1     CPU          16       SGD          0.001  resnet50              81.41   
2     CPU          16      Adam          0.001  resnet18              87.54   
3     CPU          16      Adam          0.001  resnet50              83.62   
4     GPU          16       SGD          0.001  resnet18              85.81   
5     GPU          16       SGD          0.001  resnet50              78.65   
6     GPU          16      Adam          0.001  resnet18              87.00   
7     GPU          16      Adam          0.001  resnet50              82.55   

   Train Time (ms)  FLOPs (MFLOPs)  
0       1417608.87          142.04  
1       3153081.40          329.06  
2       2023533.34          142.04  
3       4042739.54          329.06  
4       1317546.80        

In [15]:

# for batch size 32

def estimate_flops(model, input_size=(1, 3, 64, 64)):
    try:
        from fvcore.nn import FlopCountAnalysis
        dummy = torch.randn(input_size).to(next(model.parameters()).device)
        flops = FlopCountAnalysis(model, dummy).total()
        return round(flops / 1e6, 2)
    except:
        return None


def train_with_timing(model, train_loader, val_loader, optimizer, device, epochs=2):
    start = time.time()

    train_model(
        model,
        train_loader,
        val_loader,
        optimizer,
        epochs=epochs,
        device=device
    )

    end = time.time()
    return round((end - start) * 1000, 2)

def run_q2_experiments(epochs=2):
    configs = [
        ("CPU",  "SGD",  0.001),
        ("CPU",  "Adam", 0.001),
        ("GPU",  "SGD",  0.001),
        ("GPU",  "Adam", 0.001),
    ]

    results = []

    for compute, opt_name, lr in configs:
        device_used = torch.device(
            "cuda" if compute == "GPU" and torch.cuda.is_available() else "cpu"
        )

        print(f"\nRunning on {compute} with {opt_name}")

        train_loader, val_loader, test_loader = get_dataloaders(
            "FashionMNIST", batch_size=16
        )

        for model_name in ["resnet18", "resnet50"]:

            # No pretrained weights
            model = models.resnet18(weights=None) if model_name == "resnet18" \
                    else models.resnet50(weights=None)

            # Modify the first convolutional layer to accept 1 input channel for grayscale images
            model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
            model.fc = nn.Linear(model.fc.in_features, 10)

            optimizer = (
                optim.SGD(model.parameters(), lr=lr)
                if opt_name == "SGD"
                else optim.Adam(model.parameters(), lr=lr)
            )

            # Measure training time
            train_time = train_with_timing(
                model,
                train_loader,
                val_loader,
                optimizer,
                device=device_used,
                epochs=epochs
            )

            # Accuracy
            test_acc = evaluate_model(model, test_loader, device=device_used)

            # FLOPs
            flops = estimate_flops(model, input_size=(1, 1, 64, 64)) # Adjusted input size for FLOPs

            results.append({
                "Compute": compute,
                "Batch Size": 32,
                "Optimizer": opt_name,
                "Learning Rate": lr,
                "Model": model_name,
                "Test Accuracy (%)": round(test_acc, 2),
                "Train Time (ms)": train_time,
                "FLOPs (MFLOPs)": flops
            })

            print(f"{model_name} | Acc={test_acc:.2f}% | Time={train_time} ms | FLOPs={flops}")

    return pd.DataFrame(results)




In [ ]:
df_q2 = run_q2_experiments(epochs=2)
print("\nFinal Q2 Results:")
print(df_q2)


Running on CPU with SGD
Epoch 1/2, Train Loss: 0.6207, Val Loss: 0.4000, Val Accuracy: 85.45%
Epoch 2/2, Train Loss: 0.4085, Val Loss: 0.3614, Val Accuracy: 86.78%


resnet18 | Acc=85.96% | Time=1339762.81 ms | FLOPs=142.04
